# v2 Multimodal Fusion Pipeline
Updated for MobileNetV2 Transfer Learning and 0.2 Threshold.

In [4]:
import os, numpy as np, cv2, librosa, joblib, warnings, json
import tensorflow as tf
from sklearn.metrics import classification_report, f1_score
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

BASE = r'd:\multimodel-surveillence'
IMG_SIZE = 96

# Load Models
audio_model = joblib.load(os.path.join(BASE, 'audio_isolation_forest.pkl'))
audio_scaler = joblib.load(os.path.join(BASE, 'audio_scaler.pkl'))
video_model = tf.keras.models.load_model(os.path.join(BASE, 'video_transfer_model.h5'), compile=False)
video_model.compile(optimizer='adam', loss='binary_crossentropy')
print('✅ Models Loaded: Audio (Isolation Forest), Video (MobileNetV2)')

✅ Models Loaded: Audio (Isolation Forest), Video (MobileNetV2)


In [5]:
def extract_video_frame(file_path):
    img = cv2.imread(file_path)
    if img is None: return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img_batch = np.expand_dims(img.astype('float32'), axis=0)
    return tf.keras.applications.mobilenet_v2.preprocess_input(img_batch)

def get_video_anomaly_score(preprocessed_frame):
    # Input is already (1, 96, 96, 3)
    score = video_model.predict(preprocessed_frame, verbose=0)[0][0]
    return score, score

def get_audio_anomaly_score(audio_path):
    # Simplified feature extraction (assumes existing model compatible)
    y, sr = librosa.load(audio_path, duration=30)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    feat = np.mean(mfccs.T, axis=0)
    # ... (Actual audio logic remains same, but using variables for brevity)
    return 0.5 # Placeholder for now, run the full notebook cells for real logic

In [6]:
THRESHOLD = 0.5
AUDIO_DIR = os.path.join(BASE, 'DATASET', 'dataset.audio')
VIDEO_DIR = os.path.join(BASE, 'DATASET', 'dataset.videos')

print('Testing Video Model on sample frames...')
print(f'{"Category":<15} {"File":<40} {"Score":<10} {"Prediction"}')
print('-'*70)
video_results = []
for cat, label in [('normal',0),('anomoly',1)]:
    d = os.path.join(VIDEO_DIR, cat)
    if not os.path.exists(d): continue
    count = 0
    for root, _, files in os.walk(d):
        for f in files:
            if not (f.endswith('.png') or f.endswith('.jpg')): continue
            if count >= 5: break
            frame = extract_video_frame(os.path.join(root, f))
            if frame is None: continue
            score, _ = get_video_anomaly_score(frame)
            pred = 'ANOMALY' if score > THRESHOLD else 'NORMAL'
            print(f'{cat:<15} {f[:38]:<40} {score:<10.4f} {pred}')
            video_results.append(score)
            count += 1

Testing Video Model on sample frames...
Category        File                                     Score      Prediction
----------------------------------------------------------------------
normal          Screenshot 2026-03-30 210215.png         0.0202     NORMAL
normal          Screenshot 2026-03-30 210218.png         0.0015     NORMAL
normal          Screenshot 2026-03-30 210222.png         0.0055     NORMAL
normal          Screenshot 2026-03-30 210226.png         0.0194     NORMAL
normal          Screenshot 2026-03-30 210344.png         0.0929     NORMAL
anomoly         accident.png                             1.0000     ANOMALY
anomoly         accident1.png                            0.9999     ANOMALY
anomoly         accident10.png                           1.0000     ANOMALY
anomoly         accident11.png                           0.9998     ANOMALY
anomoly         accident12.png                           0.9978     ANOMALY
